## Pruebas finales del código de `getting_data.ipynb`

### 1. Obtención de los catálogos

In [15]:
# ======================================================================================
# OBTENCIÓN DE DATOS (VERSIÓN FINAL Y CORREGIDA)
# ======================================================================================

import os
import json
import csv
import time

from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.chrome.options import Options


# ======================================================================================
# DRIVER
# ======================================================================================

def crear_driver():
    """Crea y devuelve una instancia de Chrome en modo headless."""

    opciones = Options()
    opciones.add_argument("--headless")
    opciones.add_argument("--no-sandbox")
    opciones.add_argument("--disable-dev-shm-usage")
    return webdriver.Chrome(options=opciones)


# Variable global del driver, se inicializa en control_flujo()
driver = None


# ======================================================================================
# BÚSQUEDA DE CATÁLOGO: EDITORIAL NORMAL (por páginas)
# ======================================================================================

def buscar_editorial(id_editorial, nombre_editorial, pag_inicio, pag_fin):
    """
    Recorre el catálogo de una editorial normal (hasta 200 páginas) iterando de pag_inicio a pag_fin.
    Devuelve los diccionarios con título, autor, precio y URL de cada libro.

    Parámetros:
    * **id_editorial:** el id asignado a la editorial
    * **nombre_editorial:** el nombre de la editorial
    * **pag_inicio:** página del catálogo web donde empezar a hacer scraping
    * **pag_final:** última página del catálogo donde hacer scraping

    Outputs: 
    * Lista de diccionarios con los resultados del scraping
    """
    libros = []
    num_pagina = pag_inicio

    while num_pagina <= pag_fin:
        url = f"https://www.todostuslibros.com/editoriales/{id_editorial}/catalogo?page={num_pagina}"

        # Mostrar progreso solo hasta la página 10 para no llenar la consola
        if num_pagina < 10:
            print(f"Página {num_pagina}...")
        elif num_pagina == 10:
            print("Página 10 y más...")

        driver.get(url)
        time.sleep(1)

        soup = BeautifulSoup(driver.page_source, "html.parser")
        elementos = soup.select("h2 a")

        # Si no hay libros en la página, se acabó el catálogo
        if not elementos:
            print("Sin más resultados.")
            break

        for h2 in soup.select("h2"):
            a = h2.find("a")
            if not a:
                continue

            titulo = a.get_text(strip=True)

            # El autor está en el h3 inmediatamente después del h2
            h3 = h2.find_next_sibling("h3")
            autor = h3.get_text(strip=True) if h3 else ""

            # El precio está en el primer <strong> después del h2
            etiqueta_precio = h2.find_next("strong")
            precio = etiqueta_precio.get_text(strip=True) if etiqueta_precio else ""

            url_libro = a["href"] if a.get("href") else ""

            libros.append({
                "editorial": nombre_editorial,
                "titulo": titulo,
                "autor": autor,
                "precio": precio,
                "url": url_libro,
            })

        num_pagina += 1
        time.sleep(0.5)

    return libros


# ======================================================================================
# BÚSQUEDA DE CATÁLOGO: EDITORIAL GRANDE (por años, para superar el límite de 200 págs)
# ======================================================================================

def buscar_editorial_grande(id_editorial, nombre_editorial, anio_inicio, anio_fin):
    """
    Recorre el catálogo de una editorial grande filtrando por año, lo que permite superar el límite de 200 páginas por búsqueda.
    Para cada año itera todas las páginas disponibles hasta que no haya resultados.
    Devuelve los diccionarios con título, autor, precio y URL de cada libro.

    Parámetros:
    * **id_editorial:** el id asignado a la editorial
    * **nombre_editorial:** el nombre de la editorial
    * **anio_inicio:** página (del año) del catálogo web donde empezar a hacer scraping
    * **anio_final:** última página (del año) del catálogo donde hacer scraping

    Outputs: 
    * Lista de diccionario con los resultados del scraping
    """
    libros = []

    for anio in range(anio_inicio, anio_fin + 1):
        print(f"Año {anio}...")
        num_pagina = 1

        while True:
            url = f"https://www.todostuslibros.com/editoriales/{id_editorial}/catalogo?anios={anio}&page={num_pagina}"
            driver.get(url)
            time.sleep(1)

            soup = BeautifulSoup(driver.page_source, "html.parser")
            elementos = soup.select("h2 a")
            
            # Si no hay libros, este año ya no tiene más páginas
            if not elementos:
                print(f"Sin más resultados en {anio}, página {num_pagina}.")
                break

            for h2 in soup.select("h2"):
                a = h2.find("a")
                if not a:
                    continue

                titulo = a.get_text(strip=True)

                h3 = h2.find_next_sibling("h3")
                autor = h3.get_text(strip=True) if h3 else ""

                etiqueta_precio = h2.find_next("strong")
                precio = etiqueta_precio.get_text(strip=True) if etiqueta_precio else ""

                url_libro = a["href"] if a.get("href") else ""

                libros.append({
                    "editorial": nombre_editorial,
                    "titulo": titulo,
                    "autor": autor,
                    "precio": precio,
                    "url": url_libro,
                })

            num_pagina += 1
            time.sleep(0.5)

    return libros


# ======================================================================================
# EXTRACCIÓN DE FICHA TÉCNICA Y SINOPSIS
# ======================================================================================

def extraer_datos(url):
    """
    Accede a la ficha de un libro y extrae todos los datos técnicos (ISBN, páginas, formato, etc.) y la sinopsis completa.
    Devuelve un diccionario con todos los campos encontrados.

    Parámetros:
    * **url:** link de la página de la ficha técnica

    Outputs: 
    * Diccionario con los resultados del scraping
    """
    driver.get(url)
    time.sleep(2)

    soup = BeautifulSoup(driver.page_source, "html.parser")

    # La ficha técnica está dentro de elementos <dl class="datos-tecnicos">
    secciones = soup.find_all("dl", class_="datos-tecnicos")

    nombres = []  # nombres de los campos (dt)
    datos = []    # valores de los campos (dd)

    for seccion in secciones:
        etiquetas_nombre = seccion.find_all("dt")
        etiquetas_dato = seccion.find_all("dd")

        # Extraer nombres de los campos técnicos
        for nombre in etiquetas_nombre:
            nombres.append(nombre.get_text(strip=True).replace(":", "").strip())

        # Extraer valores de los campos técnicos
        # Hay tres posibles estructuras dentro de cada <dd>:
        for dato in etiquetas_dato:

            # Caso 1: el valor está en uno o varios <a> (ej: categorías, editorial)
            enlaces = dato.find_all("a")
            if enlaces:
                lista_valores = [enlace.get_text(strip=True) for enlace in enlaces]
                # Si hay un solo valor lo guardamos como string, si hay varios como lista
                datos.append(lista_valores[0] if len(lista_valores) == 1 else lista_valores)
                continue

            # Caso 2: el valor está en un <span> (ej: idioma)
            span = dato.find("span")
            if span:
                datos.append(span.get_text(strip=True))
                continue

            # Caso 3: el valor está directamente en el <dd> (ej: dimensiones, páginas)
            # Usamos split/join para limpiar espacios y saltos de línea extra
            texto = dato.get_text()
            datos.append(" ".join(texto.split()))

    # Sinopsis completa: está en un div separado fuera de la ficha técnica
    sinopsis = soup.find("div", id="collapseSynopsis")
    nombres.append("Sinopsis")
    if sinopsis:
        # Extraemos párrafo a párrafo y los unimos con " | "
        parrafos = [p.get_text(strip=True) for p in sinopsis.find_all("p")]
        datos.append(" | ".join(parrafos))
    else:
        datos.append("Sin sinopsis")

    # Construir diccionario emparejando cada nombre con su dato
    ficha_tecnica = {}
    for nombre, dato in zip(nombres, datos):
        ficha_tecnica[nombre] = dato

    return ficha_tecnica


# ======================================================================================
# SCRAPING COMPLETO DE UNA EDITORIAL (búsqueda + fichas + guardado CSV)
# ======================================================================================

def scrapear_editorial(id_editorial, nombre_editorial, inicio, fin, es_grande):
    """
    Orquesta el proceso completo para una editorial:
    1. Busca todos los libros del catálogo en el intervalo indicado.
    2. Entra en cada ficha técnica y extrae los datos.
    3. Guarda el resultado en un CSV en la carpeta 'data/'.

    Parámetros:
    * **id_editorial:** identificador de la URL (ej: "debolsillo_179709")
    * **nombre_editorial:** nombre legible (ej: "DEBOLSILLO")
    * **inicio:** página o año de inicio
    * **fin:** página o año de fin
    * **es_grande:** True si es editorial grande (búsqueda por años)

    Output:
    * Archivo .csv con las fichas técnicas de los libros
    """
    print(f"\n{'='*60}")
    print(f"Editorial: {nombre_editorial}")
    print(f"{'='*60}")

    # Fase 1: recoger listado de libros
    print("Leyendo catálogo...")
    if es_grande:
        libros = buscar_editorial_grande(id_editorial, nombre_editorial, inicio, fin)
    else:
        libros = buscar_editorial(id_editorial, nombre_editorial, inicio, fin)

    print(f"✓ {len(libros)} libros encontrados.")

    if not libros:
        print("No hay libros que procesar.")
        return

    # Fase 2: extraer fichas técnicas
    print("Extrayendo fichas técnicas...")
    lista_libros = []

    for i, libro in enumerate(libros):
        print(f"  [{i+1}/{len(libros)}] {libro['titulo'][:50]}...")

        ficha = extraer_datos(libro["url"] + "#fichaTecnica")

        # Añadir campo traducción si no existe
        if "Traducción" not in ficha:
            ficha["Traducción"] = "Sin traducción"

        # Añadir datos del catálogo a la ficha
        ficha["Título"] = libro["titulo"]
        ficha["Precio"] = libro["precio"]
        ficha["URL"] = libro["url"]

        lista_libros.append(ficha)

    # Fase 3: guardar JSON
    os.makedirs("data", exist_ok=True)
    ruta_json = f"data/bronze/catalogos/catalogo_{nombre_editorial.lower()}.json"
    campos = lista_libros[0].keys()

    # Revisión del catálogo
    if os.path.exists(ruta_json):
        with open(ruta_json, "r", encoding="utf-8") as f:
            libros_existentes = json.load(f)
    else:
        libros_existentes = []

    # Añadir nuevos libros al final
    libros_existentes.extend(lista_libros)

    # Guardar el resultado completo
    with open(ruta_json, "w", encoding="utf-8") as f:
        json.dump(libros_existentes, f, ensure_ascii=False, indent=2)

    print(f"✓ {len(lista_libros)} fichas guardadas en {ruta_json}.")


# ======================================================================================
# CONTROL DE FLUJO INTERACTIVO
# ======================================================================================

def control_flujo(ruta_editoriales="data/json/editoriales.json", ruta_estado="data/json/estado.json"):
    """
    Función principal que gestiona el flujo interactivo del scraping.

    Lee el fichero de editoriales (editoriales.json) con la información de cada una,
    consulta el estado guardado (estado.json) para saber cuáles ya están procesadas,
    y pregunta al usuario qué hacer con cada una.

    Estructura esperada de editoriales.json:
    {
        "debolsillo": {
            "id": "debolsillo_179709",
            "grande": false,
            "intervalo_max": [1, 200]   <- páginas si normal, años si grande
        },
        "espasa": {
            "id": "espasa_76490",
            "grande": true,
            "intervalo_max": [1990, 2024]
        }
    }

    Estructura de estado.json (se genera automáticamente):
    {
        "debolsillo": {
            "ultimo": 200    <- última página/año procesada
        }
    }
    """
    global driver

    # Cargar información de editoriales
    if not os.path.exists(ruta_editoriales):
        print(f"Error: no se encuentra '{ruta_editoriales}'.")
        return

    with open(ruta_editoriales, "r", encoding="utf-8") as f:
        info_editoriales = json.load(f)

    # Cargar estado previo si existe, o empezar desde cero
    if os.path.exists(ruta_estado):
        with open(ruta_estado, "r", encoding="utf-8") as f:
            estado = json.load(f)
    else:
        estado = {}

    # Iniciar el driver una sola vez para toda la sesión
    print("Iniciando navegador...")
    driver = crear_driver()
    print("✓ Navegador listo.\n")

    try:
        for nombre, info in info_editoriales.items():
            maximo = info["intervalo"][1]
            es_grande = info["grande"]
            id_editorial = info["id"]

            # Comprobar si ya está completamente procesada
            if nombre in estado and estado[nombre]["ultimo"] >= maximo:
                print(f"{nombre}: ya procesada completamente, saltando...")
                continue

            # Determinar punto de inicio (desde el principio o desde donde se dejó)
            if nombre in estado:
                ultimo_guardado = estado[nombre]["ultimo"]
                inicio_sugerido = ultimo_guardado+1 # retomamos desde el último guardado
                msg = f"{nombre}: proceso iniciado. Último guardado en {ultimo_guardado}. ¿Continuar? [Y/N]: "
            else:
                inicio_sugerido = info["intervalo"][0]
                msg = f"{nombre}: aún no procesada. ¿Comenzar? [Y/N]: "

            respuesta = input(msg).strip().upper()

            if respuesta == "N":
                print(f"Saltando {nombre}.\n")
                continue

            elif respuesta == "Y":
                # Pedir intervalo al usuario
                tipo = "año" if es_grande else "página"
                print(f"Inicio sugerido: {inicio_sugerido} | Máximo disponible: {maximo}")

                entrada_inicio = input(f"Introduce {tipo} de inicio [{inicio_sugerido}]: ").strip()
                entrada_fin = input(f"Introduce {tipo} de fin (máx. {maximo}): ").strip()

                # Usar valores sugeridos si el usuario no introduce nada
                inicio = int(entrada_inicio) if entrada_inicio else inicio_sugerido
                fin = min(int(entrada_fin), maximo)  # nunca superar el máximo

                # Ejecutar el scraping
                scrapear_editorial(id_editorial, nombre, inicio, fin, es_grande)

                # Actualizar y guardar estado
                if nombre not in estado:
                    estado[nombre] = {}
                estado[nombre]["ultimo"] = fin

                with open(ruta_estado, "w", encoding="utf-8") as f:
                    json.dump(estado, f, ensure_ascii=False, indent=2)

                print(f"Estado guardado: {nombre} → hasta {tipo} {fin}.\n")

            else:
                print("Respuesta no válida, saltando.\n")

    finally:
        # Cerrar el navegador siempre, aunque haya errores
        driver.quit()
        print("\nNavegador cerrado.")


# ======================================================================================
# PUNTO DE ENTRADA
# ======================================================================================

# if __name__ == "__main__":
#     control_flujo()

In [18]:
# ======================================================================================
# NUEVO SCRAPING
# ======================================================================================

def buscar_editorial(id_editorial, nombre_editorial, pag_inicio, pag_fin):
    """
    Recorre el catálogo de una editorial normal (hasta 200 páginas) iterando de pag_inicio a pag_fin.
    Devuelve los diccionarios con título, autor, precio y URL de cada libro.

    Parámetros:
    * **id_editorial:** el id asignado a la editorial
    * **nombre_editorial:** el nombre de la editorial
    * **pag_inicio:** página del catálogo web donde empezar a hacer scraping
    * **pag_final:** última página del catálogo donde hacer scraping

    Outputs: 
    * Lista de diccionarios con los resultados del scraping
    """
    libros = []
    num_pagina = pag_inicio

    while num_pagina <= pag_fin:
        url = f"https://www.todostuslibros.com/editoriales/{id_editorial}/catalogo?page={num_pagina}"

        # Mostrar progreso solo hasta la página 10 para no llenar la consola
        if num_pagina < 10:
            print(f"Página {num_pagina}...")
        elif num_pagina == 10:
            print("Página 10 y más...")

        driver.get(url)
        time.sleep(1)

        soup = BeautifulSoup(driver.page_source, "html.parser")
        elementos = soup.find_all("article", class_="book-col")

        for el in elementos:
            titulo = el.select_one("p.title").get_text(strip=True)
            autor = el.select_one("p.author").get_text(strip=True)
            precio = el.select_one("div.prices").get_text(strip=True)
            url_libro = el.select_one("p.title a")['href']
        

            libros.append({
                "editorial": nombre_editorial,
                "titulo": titulo,
                "autor": autor,
                "precio": precio,
                "url": url_libro,
            })

        num_pagina += 1
        time.sleep(0.5)

    return libros


# ======================================================================================
# EDITORIAL GRANDE NUEVO
# ======================================================================================

def buscar_editorial_grande(id_editorial, nombre_editorial, anio_inicio, anio_fin):
    """
    Recorre el catálogo de una editorial grande filtrando por año, lo que permite superar el límite de 200 páginas por búsqueda.
    Para cada año itera todas las páginas disponibles hasta que no haya resultados.
    Devuelve los diccionarios con título, autor, precio y URL de cada libro.

    Parámetros:
    * **id_editorial:** el id asignado a la editorial
    * **nombre_editorial:** el nombre de la editorial
    * **anio_inicio:** página (del año) del catálogo web donde empezar a hacer scraping
    * **anio_final:** última página (del año) del catálogo donde hacer scraping

    Outputs: 
    * Lista de diccionario con los resultados del scraping
    """
    libros = []

    for anio in range(anio_inicio, anio_fin + 1):
        print(f"Año {anio}...")
        num_pagina = 1

        while True:
            url = f"https://www.todostuslibros.com/editoriales/{id_editorial}/catalogo?anios={anio}&page={num_pagina}"
            driver.get(url)
            time.sleep(1)

            soup = BeautifulSoup(driver.page_source, "html.parser")
            elementos = soup.find_all("article", class_="book-col")
    
            for el in elementos:
                titulo = el.select_one("p.title").get_text(strip=True)
                autor = el.select_one("p.author").get_text(strip=True)
                precio = el.select_one("div.prices").get_text(strip=True)
                url_libro = el.select_one("p.title a")['href']
        
    
                libros.append({
                    "editorial": nombre_editorial,
                    "titulo": titulo,
                    "autor": autor,
                    "precio": precio,
                    "url": url_libro,
                })

            num_pagina += 1
            time.sleep(0.5)

    return libros


control_flujo()

Iniciando navegador...
✓ Navegador listo.

Planeta: ya procesada completamente, saltando...
Espasa: ya procesada completamente, saltando...
Seix Barral: ya procesada completamente, saltando...
Saltando Destino.

Booket: ya procesada completamente, saltando...
Tusquets: ya procesada completamente, saltando...
Alfaguara: ya procesada completamente, saltando...
Saltando Plaza & Janes.

Debolsillo: ya procesada completamente, saltando...
Anagrama: ya procesada completamente, saltando...
Editorial Anagrama: ya procesada completamente, saltando...
Acantilado: ya procesada completamente, saltando...
Siruela: ya procesada completamente, saltando...
Tusquets Editores: ya procesada completamente, saltando...
Maxi Tusquets: ya procesada completamente, saltando...
Ediciones Akal: ya procesada completamente, saltando...
Gredos: ya procesada completamente, saltando...
Alianza Editorial: ya procesada completamente, saltando...
Ediciones Cátedra: ya procesada completamente, saltando...
RAE: ya procesa

Fin de capa **bronze**

### 2. Creación del DataFrame base para guardar en parquet

In [3]:
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime
import re
import os 
import json
from src.constants import TRADUCTOR_EDITOR, OTROS_CONTRIBUIDORES, ILUSTRACIONES, ESCOLARES, CATEGORIAS, COLUMNAS_FINALES, CATEGORIAS, SUBCATEGORIAS, ENCUADERNACION
from src.utils import leer_json

# ======================================================================================
# CREACIÓN DEL DATAFRAME BASE
# ======================================================================================

def crear_df(ruta_catalogos="data/bronze/catalogos"):
    path = Path(ruta_catalogos)
    df = pd.DataFrame({})
    jsons = []

    print("="*50,"\nCreando DataFrame con todos los libros\n","="*50)
    for archivo in path.iterdir():
            print(f"Añadiendo {archivo.name}")
            editorial = leer_json(archivo, df=True)
            jsons.append(editorial)

    df = pd.concat(jsons, axis=0)

    # Borrar filas repetidas
    print("Catálogos convertidos a DataFrame. Eliminando filas duplicadas...")
    df.drop_duplicates(subset=['EAN'], keep='first', inplace=True)

    # Limpiado de nombre de columnas
    df.columns = df.columns.str.strip().str.lower().str.translate(str.maketrans({"á": "a", "é": "e", "í": "i", "ó":"o", "ú": "u", "º": "", " ": "_"}))

    print("DataFrame creado con éxito.")
    return df


# =============================================================================
# FUNCIONES AUXILIARES Y LIMPIEZA BÁSICA
# =============================================================================

# Búsqueda de la moda en una lista de strings
def moda(x):
    """
    Devuelve la moda de una serie.
    """

    x = x.dropna()

    if len(x) == 0:
        return np.nan

    return x.mode().iloc[0]


# Extrae el número (precio, medida...) de un string
def extraer_numero(x):

    _NUMERO = re.compile(r"(\d+[.,]?\d*)")

    if pd.isna(x):
        return np.nan

    m = _NUMERO.search(str(x))

    if m is None:
        return np.nan

    return float(m.group(1).replace(",", "."))


# Conversión de los elementos de una lista/columna en listas
def normalizar_lista(valor):
    """
    Convierte cualquier valor en una lista.

    NaN -> []
    str -> [str]
    list -> list limpia
    ndarray -> list
    """

    if valor is None:
        return []

    if isinstance(valor, float) and np.isnan(valor):
        return []

    if isinstance(valor, str):
        valor = valor.strip().title()
        if valor == "":
            return []

        return [valor]

    if isinstance(valor, np.ndarray):
        valor = valor.tolist()

    if isinstance(valor, (list, tuple)):
        salida = []
        for x in valor:
            if pd.isna(x):
                continue

            x = str(x).strip().title()

            if x:
                salida.append(x)

        return list(dict.fromkeys(salida))

    return [str(valor)]


def normalizar_columnas_lista(df, columnas_listas):

    df = df.copy()

    for col in columnas_listas:
        if col in df.columns:
            df[col] = df[col].apply(normalizar_lista)

    return df


# Limpieza básica del DF (eliminar filas son datos obligatorios, duplicados, relleno de columnas nulas y mapeo)
def limpieza_basica(df,dict_editoriales=None,dict_encuadernacion=ENCUADERNACION):

    # quitar duplicados por EAN
    df.drop_duplicates(subset="ean", inplace=True)

    # eliminar libros sin autor y sin categoría
    df = df.dropna(
        subset=["ean", "titulo", "autoria", "categorias"],
        how="any",
    )
    df['ean'] = df['ean'].astype(str)

    # fecha
    df["fecha_publicacion"] = pd.to_datetime(
        df["fecha_publicacion"],
        format="%d-%m-%Y",
        errors="coerce",
    )

    # sinopsis
    df["sinopsis"] = df["sinopsis"].fillna("Sin sinopsis")

    # ids editoriales
    if dict_editoriales is not None:
        df["editorial"] = df["editorial"].map(dict_editoriales)

    # encuadernación
    if dict_encuadernacion is not None:
        df["encuadernacion"] = df["encuadernacion"].map(dict_encuadernacion)

    return df

def normalizar_titulos(nombre:str):
    articulos = {"El", "La", "Los", "Las", "Un", "Una", "Unos", "Unas"}

    if "," in nombre:
        titulo, articulo = map(str.strip, nombre.rsplit(",", 1))
        if articulo in articulos:
            nombre = f"{articulo} {titulo}"

    nombre = nombre.strip().title()

    return nombre 

# =============================================================================
# MERGE DE COLUMNAS Y FEATURES
# =============================================================================

# Unión de columnas para crear otra nueva
def merge_columnas(df, nombre, columnas):

    df = df.copy()

    for col in columnas:
        if col not in df.columns:
            df[col] = [[] for _ in range(len(df))]

    df[nombre] = df[columnas].sum(axis=1).apply(lambda x: list(dict.fromkeys(x)))

    return df

# Coversión de columnas numéricas que aparecen como str
def extraer_numeros(df):

    df = df.copy()

    for col in ["precio","peso","grueso","n_paginas"]:
        if col in df.columns:
            df[col] = df[col].apply(extraer_numero)

    # dimensiones (ej: 240 x 170 mm)
    medidas = df["dimensiones"].astype(str).str.extract(r"(\d+[.,]?\d*)\D+(\d+[.,]?\d*)")

    df["alto_mm"] = medidas[0].str.replace(",", ".", regex=False).astype(float)
    df["ancho_mm"] = medidas[1].str.replace(",", ".", regex=False).astype(float)

    return df

def crear_marcadores(df, ilustraciones, escolares):
    # Escolares
    escolar = (
        df[escolares]
        .apply(lambda col: col.str.len())
        .sum(axis=1)
        > 0
    )
    df['es_escolar'] = escolar 

    # Ilustrada
    ilustrada = (
        df[ilustraciones]
        .apply(lambda col: col.str.len())
        .sum(axis=1)
        > 0
    )
    df['es_ilustrada'] = ilustrada

    # Impresión bajo demanda
    ibd = (
        df["ibd"]
        .str.len()
        > 0
    )
    df['es_ibd'] = ibd

    return df


def crear_portada(df:pd.DataFrame):
    # URL imagen
    ean = df["ean"].astype(str)
    df["img"] = (
        "https://static.cegal.es/imagenes/marcadas/"
        + ean.str[:8]
        + "/"
        + ean
        + ".gif"
    )

    return df


def definir_aparato_critico(df, cols_ap):

    def tiene_contenido(valor):
        if valor is None:
            return False
        if isinstance(valor, float) and pd.isna(valor):
            return False
        if isinstance(valor, (list, np.ndarray)) and len(valor) == 0:
            return False
        return True

    # Creamos la máscara booleana
    mask = df[cols_ap].map(tiene_contenido).any(axis=1)
    df["aparato_critico"] = mask

    # Generamos los valores directamente en el apply sin necesidad de .loc
    def extraer_tipos(fila):
        presentes = [col for col in cols_ap if tiene_contenido(fila[col])]
        return presentes if presentes else np.nan

    df["tipo_aparato_critico"] = df[cols_ap].apply(extraer_tipos, axis=1)

    return df

def rellenar_columnas(df):

    df = df.copy()

    # Medidas por editorial + colección
    for col in ["alto_mm", "ancho_mm", "precio", "n_paginas"]:
        mediana_col = df.groupby(["editorial", "coleccion"])[col].transform("median")
        mediana_enc = df.groupby(["editorial", "encuadernacion"])[col].transform("median")
        
        df[col] = df[col].fillna(mediana_col).fillna(mediana_enc)


    # Grosor (fórmula estándar)
    df["grueso"] = df["grueso"].fillna(df["n_paginas"] * 0.04)

    # Peso (fórmula estándar)
    peso_estimado = df['peso'].fillna((df["alto_mm"]/1000) * (df["ancho_mm"]/1000) * (df["n_paginas"]/2) * 80 + 120)

    df["peso"] = df["peso"].fillna(peso_estimado)

    # Idioma original 
    df["autor_principal"] = (
        df["autoria"]
        .apply(
            lambda x:
                x[0]
                if len(x)
                else np.nan
        )
    )

    idioma = (
        df.groupby("autor_principal")[
            "idioma_original"
        ]
        .transform(moda)
    )

    df["idioma_original"] = df["idioma_original"].fillna(idioma)

    df.drop(columns="autor_principal", inplace=True)

    return df


# inferencia categorías
def inferencia_categoria(df, categorias=SUBCATEGORIAS):

    def obtener_categorias(subcategorias_libro):
        if not isinstance(subcategorias_libro, (list, tuple, set)):
            return []

        return list({
            categoria
            for subcategoria in subcategorias_libro
            for categoria, subcategorias in categorias.items()
            if subcategoria in subcategorias
        })

    df["categorias"] = df["subcategorias"].apply(obtener_categorias)

    return df


# =============================================================================
# LIMPIEZA FINAL
# =============================================================================

def limpiar_columnas(df, cols_borrar):

    borrar = [c for c in cols_borrar if c in df.columns]

    return df.drop(columns=borrar)


# =============================================================================
# PIPELINE
# =============================================================================

def limpiar_df_completa(data, ruta_editoriales="data/json/editoriales.json", dict_encuadernacion=ENCUADERNACION):

    df = data.copy()
    if os.path.exists(ruta_editoriales):
        with open(ruta_editoriales, "r", encoding="utf-8") as f:
            dict_editoriales = json.load(f)

    TTL_A_ED = {
        nombre_ttl: editorial
        for editorial, datos in dict_editoriales.items()
        for nombre_ttl in datos["nombre_ttl"]
    }

    # Cambio de nombres de columnas
    columnas_lista = list(set(
        TRADUCTOR_EDITOR
        + OTROS_CONTRIBUIDORES
        + ILUSTRACIONES
        + ESCOLARES
        + CATEGORIAS
        + ["autoria"]
    ))
    df = normalizar_columnas_lista(df, columnas_lista)

    # Limpieza
    df = limpieza_basica(df, TTL_A_ED, dict_encuadernacion)
    df['titulo'] = df['titulo'].apply(normalizar_titulos)

    # Merge colaboradores
    df = merge_columnas(df, "traductor_y_editor", TRADUCTOR_EDITOR)
    df = merge_columnas(df,"otros_contribuidores", OTROS_CONTRIBUIDORES)
    df = merge_columnas(df, "subcategorias", CATEGORIAS)

    # Feature engineering
    df = extraer_numeros(df)
    df = definir_aparato_critico(df, OTROS_CONTRIBUIDORES)
    df = crear_marcadores(df, ESCOLARES, ILUSTRACIONES)
    df = crear_portada(df)

    # Relleno
    df = rellenar_columnas(df)
    df = inferencia_categoria(df)

    # Limpieza final
    borrar = (
        TRADUCTOR_EDITOR
        + OTROS_CONTRIBUIDORES
        + ILUSTRACIONES
        + ['isbn']
    )
    df = limpiar_columnas(df, borrar)

    df = df[[c for c in COLUMNAS_FINALES if c in df.columns]]

    return df


def validar_catalogo(df):
    # EAN únicos
    if df['ean'].nunique().count() < len(df['ean']):
        print('Existen números EAN repetidos.') 
    
    # sin nulos en columnas obligatorial
    cols_nulos = ["ean", "titulo", "autoria", "categorias"]
    for col in cols_nulos:
        if df[col].isna().sum() > 0:
            print(f'Existen nulos en la columna {col}.') 

    # fechas válidas
    formato = '%d/%m/%Y'
    for fecha in df['fecha_publicacion']:
        try:
            fecha_valida = datetime.strptime(fecha, formato)
            print("Fecha correcta")
        except ValueError:
            print("Fecha inválida")

    # dimensiones positivas
    cols_numericas = ["n_paginas","precio","alto_mm","ancho_mm","grueso","peso"]
    for col in cols_numericas:
        if any(x<=0 for x in df[col]):
            print(f"La columna {col} tiene núemeros no positivos.")


### 3. Merge de los datos del SPI

In [4]:
import json
import pandas as pd 
import numpy as np 
from pathlib import Path

if os.path.exists("data/json/editoriales.json"):
    with open("data/json/editoriales.json", "r", encoding="utf-8") as f:
        dict_editoriales = json.load(f)

SPI_A_ED = {
    nombre_spi: editorial
    for editorial, datos in dict_editoriales.items()
    for nombre_spi in (
        datos["nombre_spi"]
        if isinstance(datos["nombre_spi"], list)
        else [datos["nombre_spi"]]
    )
    if nombre_spi is not None
}

def merge_spi(ruta_spi="data/bronze/spi", spi_a_ed = SPI_A_ED):
    rutas = sorted(Path(ruta_spi).glob("*.csv"))
    dfs = []
    for ruta in rutas:
        df = pd.read_csv(ruta)

        if "Editorial" not in df.columns:
            raise ValueError(f"{ruta.name} no contiene la columna 'Editorial'.")

        nombre = ruta.stem.lower().replace("clasificacion_", "")

        df = df.rename(columns={
            c: f"{c}_{nombre}"
            for c in df.columns
            if c != "Editorial"
        })
        df["Editorial"] = df["Editorial"].map(spi_a_ed)
        df = (
            df
            .groupby("Editorial", as_index=False)
            .first()
        )
        df = df.set_index("Editorial")
        dfs.append(df)
    
    df = pd.concat(dfs, axis=1, join="outer").reset_index()

    df_selection = df[df['Editorial'].isin(spi_a_ed.values())].copy()
    return df_selection

def prestigio_editorial(df):
    df = df.loc[:, ~df.columns.duplicated()].copy()

    df_prestigio = pd.DataFrame({})
    df_prestigio["editorial"] = df["Editorial"]

    for i in range(1, len(df.columns) - 1, 2):
        col_pos = df.columns[i]
        col_icee = df.columns[i + 1]

        nombre_col = f"prestigio{str(col_pos).replace('Posición', '').strip()}"

        # Extraer como Series unimodales e iloc para evitar ambigüedades
        s_pos = df.iloc[:, i]
        s_icee = df.iloc[:, i + 1]

        mask = s_pos.notna() & s_icee.notna()

        icee = s_icee
        icee_min = icee.min()
        icee_max = icee.max()
        
        if icee_max != icee_min:
            icee_norm = (icee - icee_min) / (icee_max - icee_min)
        else:
            icee_norm = pd.Series(0.0, index=df.index)

        n = s_pos.max()
        percentil = 1 - (s_pos - 1) / (n - 1) if (pd.notna(n) and n > 1) else pd.Series(1.0, index=df.index)

        df_prestigio[nombre_col] = 0.0
        
        # Asignación segura con valores indexados por la máscara
        val_calculado = 0.1 + 0.9 * (0.8 * icee_norm[mask] + 0.2 * percentil[mask])
        df_prestigio.loc[mask, nombre_col] = val_calculado

    df_prestigio = df_prestigio.fillna(0.0)

    # Se incluyen las editoriales que no están en el SPI
    nuevas_filas = {}
    for col in df_prestigio.columns:
        nuevas_filas[col] = []

    for ed_dict in dict_editoriales.items():
        ed = ed_dict[0]
        spi = ed_dict[1]['nombre_spi']

        if spi is False:
            for col in nuevas_filas.keys():
                if col == 'editorial':
                    nuevas_filas[col].append(ed)
                else: 
                    nuevas_filas[col].append(0)

    nuevas_filas = pd.DataFrame(nuevas_filas)
    df_prestigio = pd.concat([df_prestigio, nuevas_filas], ignore_index=True)

        # df_prestig.to_parquet("data/silver/prestigio_spi.parquet", engine="pyarrow")

        # añadir filas de editoriales que no están en el spi con todo cero

    return df_prestigio

Fin de la capa **silver**

### 4. Creación de datos finales

In [5]:
import pandas as pd
import numpy as np
import ast
from sklearn.preprocessing import MinMaxScaler
from src.constants import SUBCATEGORIAS

def merge_dataframes(ttl: pd.DataFrame, spi: pd.DataFrame):
    return ttl.merge(spi, how="left", on="editorial", validate="many_to_one")

def portabilidad(df):
    columnas = ["alto_mm", "ancho_mm", "grueso", "peso"]
    scaler = MinMaxScaler()

    escaladas = pd.DataFrame(
        scaler.fit_transform(df[columnas]),
        columns=columnas,
        index=df.index
    )

    # Invertir: 1 = pequeño/ligero = más portátil
    # .clip(lower=0.01) evita que un valor máximo ponga a 0 toda la media geométrica
    escaladas = (1 - escaladas).clip(lower=0.01)

    df["indice_portabilidad"] = escaladas.prod(axis=1) ** (1 / len(columnas))
    return df

def compacidad(df):
    # Evitar división por cero o por valores nulos en dimensiones
    denominador = (df['alto_mm'] * df['ancho_mm'] * df['grueso']).replace(0, np.nan)
    df['indice_compacidad'] = (df['n_paginas'] / denominador).fillna(0.0)
    return df

def prestancia(df):
    variables = ["alto_mm", "ancho_mm", "grueso", "peso"]
    scaler = MinMaxScaler()

    normalizadas = pd.DataFrame(
        scaler.fit_transform(df[variables]),
        columns=variables,
        index=df.index
    )

    pesos_encuadernacion = {
        "rústica": 0.4,
        "rústica con solapas": 0.5,
        "tapa blanda": 0.4,
        "cartoné": 0.7,
        "tapa dura": 0.8,
        "tela": 0.9,
        "piel": 1.0,
    }

    encuadernacion = (
        df["encuadernacion"]
        .fillna("")
        .astype(str)
        .str.lower()
        .str.strip()
        .map(pesos_encuadernacion)
        .fillna(0.5)
    )

    df["indice_prestancia"] = (
        0.20 * (normalizadas["alto_mm"] + normalizadas["ancho_mm"]) / 2
        + 0.15 * normalizadas["grueso"]
        + 0.20 * normalizadas["peso"]
        + 0.45 * encuadernacion
    )
    return df

def ap_critico(df, pesos=None):
    # Claves normalizadas SIN tildes para coincidir con tipo_aparato_critico
    if pesos is None:
        pesos = {
            "introduccion": 1.0,
            "prologo": 1.0,
            "epilogo": 1.0,
            "notas": 1.5,
            "anotaciones": 1.5,
            "estudio": 2.0,
            "comentarios": 1.5,
            "bibliografia": 1.0,
            "cronologia": 0.5,
            "trabajo_preliminar": 1.0
        }

    def calcular_score(textos):
        if isinstance(textos, str):
            try:
                textos = ast.literal_eval(textos)
            except (ValueError, SyntaxError):
                textos = []

        if not isinstance(textos, (list, tuple, set)):
            return 0.0

        return sum(pesos.get(str(texto).lower().strip(), 0) for texto in textos)

    df["score_critico"] = df["tipo_aparato_critico"].apply(calcular_score)
    return df

def colaboradores(df):
    def parse_lista(val):
        if isinstance(val, str):
            try:
                return ast.literal_eval(val)
            except (ValueError, SyntaxError):
                return []
        return val if isinstance(val, list) else []

    colabs_series = df["otros_contribuidores"].apply(parse_lista)
    frecuencia = colabs_series.explode().dropna().value_counts()

    def calcular_score(lista):
        if not isinstance(lista, (list, tuple, set)) or len(lista) == 0:
            return 0.0

        valores = [frecuencia.get(colaborador, 0) for colaborador in lista]
        if not valores:
            return 0.0

        return np.mean(np.log1p(valores))

    df["score_colaboradores"] = colabs_series.apply(calcular_score)
    return df

def prestigio(df):
    def parse_lista(val):
        if isinstance(val, str):
            try:
                return ast.literal_eval(val)
            except (ValueError, SyntaxError):
                return []
        return val if isinstance(val, list) else []

    def calcular_prestigio(fila):
        subcategorias = parse_lista(fila.get("subcategorias"))
        categorias = parse_lista(fila.get("categorias"))

        if not subcategorias or not categorias:
            return 0.0  # Cambiado np.nan por 0.0 para no perder la fila

        conteo = {}
        for subcategoria in subcategorias:
            for categoria in categorias:
                if categoria in SUBCATEGORIAS and subcategoria in SUBCATEGORIAS[categoria]:
                    conteo[categoria] = conteo.get(categoria, 0) + 1
                    break

        if not conteo:
            return 0.0

        total = sum(conteo.values())
        pesos = {cat: cant / total for cat, cant in conteo.items()}

        valores = []
        for categoria, peso in pesos.items():
            columna = f"prestigio_{categoria.lower().replace(' ', '_').replace('á','a').replace('é','e').replace('í','i').replace('ó','o').replace('ú','u')}"

            if columna not in df.columns:
                continue

            valor = fila[columna]
            if pd.notna(valor):
                valores.append((valor, peso))

        if not valores:
            return 0.0

        suma_pesos = sum(peso for _, peso in valores)
        return sum(valor * (peso / suma_pesos) for valor, peso in valores)

    df["prestigio_cat"] = df.apply(calcular_prestigio, axis=1)
    return df 

def crear_gold(ttl: pd.DataFrame, spi: pd.DataFrame):

    df = merge_dataframes(ttl, spi)
    df.dropna(subset=['alto_mm', 'ancho_mm', 'peso', 'grueso', 'n_paginas', 'precio'], inplace=True)
    
    if 'sinopsis' in df.columns:
        df.drop(columns=['sinopsis'], inplace=True)

    # 1. Usar 3.0 en IQR (Outliers extremos) en vez de 1.5 para no descartar casi todo el dataset
    columnas = ["alto_mm", "ancho_mm", "peso"]
    mask = pd.Series(True, index=df.index)

    for col in columnas:
        q1 = df[col].quantile(0.25)
        q3 = df[col].quantile(0.75)
        iqr = q3 - q1

        limite_inferior = max(0.0, q1 - 3.0 * iqr)
        limite_superior = q3 + 3.0 * iqr

        mask &= (df[col] >= limite_inferior) & (df[col] <= limite_superior)

    df = df[mask].copy()

    # 2. Pipeline de Métricas
    df = portabilidad(df)
    df = compacidad(df)
    df = prestancia(df)
    df = colaboradores(df)
    df = ap_critico(df)
    df = prestigio(df)

    # 3. Imputación final de seguridad para TOPSIS
    metricas_modelo = [
        'indice_portabilidad', 'indice_compacidad', 
        'indice_prestancia', 'score_colaboradores', 
        'score_critico', 'prestigio_cat'
    ]
    df[metricas_modelo] = df[metricas_modelo].fillna(0.0)

    return df

In [6]:
data = crear_df()
merge = merge_spi()

silver_ttl = limpiar_df_completa(data)
silver_spi = prestigio_editorial(merge)

gold_data = crear_gold(silver_ttl, silver_spi)
gold_data

Creando DataFrame con todos los libros
Añadiendo catalogo_acantilado.json
Añadiendo catalogo_alfaguara.json
Añadiendo catalogo_alianza editorial.json
Añadiendo catalogo_anagrama.json
Añadiendo catalogo_austral.json
Añadiendo catalogo_booket.json
Añadiendo catalogo_castalia ediciones.json
Añadiendo catalogo_crítica.json
Añadiendo catalogo_debolsillo.json
Añadiendo catalogo_destino.json
Añadiendo catalogo_ediciones akal.json
Añadiendo catalogo_ediciones cátedra.json
Añadiendo catalogo_editorial anagrama.json
Añadiendo catalogo_espasa.json
Añadiendo catalogo_gredos.json
Añadiendo catalogo_maxi tusquets.json
Añadiendo catalogo_penguin clásicos.json
Añadiendo catalogo_planeta.json
Añadiendo catalogo_plaza & janes.json
Añadiendo catalogo_rae.json
Añadiendo catalogo_seix barral.json
Añadiendo catalogo_siruela.json
Añadiendo catalogo_tusquets editores.json
Añadiendo catalogo_tusquets.json
Catálogos convertidos a DataFrame. Eliminando filas duplicadas...
DataFrame creado con éxito.


,ean,titulo,editorial,coleccion,autoria,traductor_y_editor,otros_contribuidores,aparato_critico,tipo_aparato_critico,categorias,subcategorias,idioma_original,idioma_de_publicacion,fecha_publicacion,n_paginas,precio,alto_mm,ancho_mm,grueso,peso,encuadernacion,es_escolar,es_ibd,url,img,prestigio_antropologia,prestigio_arqueologia,prestigio_bellas_artes,prestigio_biblioteconomia,prestigio_comunicacion,prestigio_derecho,prestigio_economia,prestigio_educacion,prestigio_filosofia,prestigio_general,prestigio_geografia,prestigio_historia,prestigio_literatura,prestigio_politica,prestigio_psicologia,prestigio_sociologia,indice_portabilidad,indice_compacidad,indice_prestancia,score_colaboradores,score_critico,prestigio_cat
0,9791387964306.0,El Significado De La Ópera,Acantilado,El Acantilado,[Chirstopher Wintle],[Francisco López Martín],[],False,NaN,"[Bellas Artes, Literatura]","[Música, Ensayos Literarios, Ópera, España]",Castellano,Castellano,2026-06-24,480.0,28.0,210.0,131.0,26.00,492.0,Otro,False,False,https://www.todostuslibros.com/libros/el-signi...,https://static.cegal.es/imagenes/marcadas/9791...,0.0,0.0,0.1,0.0,0.1,0.0,0.0,0.0,0.0,0.146333,0.0,0.112513,0.1,0.0,0.0,0.0,0.624450,0.000671,0.404325,0.0,0.0,0.10
1,9791387964283.0,"Viejas Verdades, Nuevos Clichés",Acantilado,El Acantilado,[Isaac Bashevis Singer],"[Mar Vidal Aparicio, David Stromberg]",[],False,NaN,"[Literatura, Filosofía]","[Filosofía Popular, Judaísmo, Ensayos Literari...",Castellano,Castellano,2026-06-23,256.0,22.0,210.0,131.0,14.00,268.0,Otro,False,False,https://www.todostuslibros.com/libros/viejas-v...,https://static.cegal.es/imagenes/marcadas/9791...,0.0,0.0,0.1,0.0,0.1,0.0,0.0,0.0,0.0,0.146333,0.0,0.112513,0.1,0.0,0.0,0.0,0.666154,0.000665,0.367783,0.0,0.0,0.05
2,9791387964177.0,"Rafael Moneo, Construir Las Ideas",Acantilado,El Acantilado,[Carmen Díez Medina],[Sin Traducción],[],False,NaN,"[Bellas Artes, Literatura]","[Arquitectura, Ensayos Literarios, España]",Castellano,Castellano,2026-06-22,736.0,38.0,210.0,131.0,29.44,751.0,Otro,False,False,https://www.todostuslibros.com/libros/rafael-m...,https://static.cegal.es/imagenes/marcadas/9791...,0.0,0.0,0.1,0.0,0.1,0.0,0.0,0.0,0.0,0.146333,0.0,0.112513,0.1,0.0,0.0,0.0,0.566300,0.000909,0.444836,0.0,0.0,0.10
3,9791387964610.0,Mirar El Mar,Acantilado,Narrativa del Acantilado,[Pablo Echart Orús],[Sin Traducción],[],False,NaN,[Literatura],"[Estilos De Vida Y Aficiones, Cuentos, Histori...",Castellano,Castellano,2026-06-17,96.0,14.0,210.0,131.0,6.00,108.0,Otro,False,False,https://www.todostuslibros.com/libros/mirar-el...,https://static.cegal.es/imagenes/marcadas/9791...,0.0,0.0,0.1,0.0,0.1,0.0,0.0,0.0,0.0,0.146333,0.0,0.112513,0.1,0.0,0.0,0.0,0.692172,0.000582,0.341778,0.0,0.0,0.10
4,9791387964269.0,Colette,Acantilado,El Acantilado,[Antoine Compagnon],[Núria Petit Fontserè],[],False,NaN,[Literatura],"[Ensayos Literarios, Biografía: Escritores, Es...",Castellano,Castellano,2026-06-17,176.0,20.0,210.0,131.0,10.00,189.0,Otro,False,False,https://www.todostuslibros.com/libros/colette_...,https://static.cegal.es/imagenes/marcadas/9791...,0.0,0.0,0.1,0.0,0.1,0.0,0.0,0.0,0.0,0.146333,0.0,0.112513,0.1,0.0,0.0,0.0,0.679329,0.000640,0.354935,0.0,0.0,0.10
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
88339,9788472235106.0,La Novela Criminal,Tusquets Editores,Cuadernos Infimos,[Roman Gubern Garriga],[Sin Traducción],[],False,NaN,[Literatura],"[Literatura: Historia Y Crítica, España]",NaN,Castellano,1970-01-01,80.0,3.0,180.0,100.0,3.20,177.6,Rústica,False,False,https://www.todostuslibros.com/libros/la-novel...,https://static.cegal.es/imagenes/marcadas/9788...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.000000,0.0,0.0,0.0,0.0,0.771801,0.001389,0.278518,0.0,0.0,0.00
88340,9788472235083.0,Santa Ava De Adis Abeba,Tusquets Editores,Cuadernos Infimos,[Cargenio Trías],[Sin Traducción],[],Fa

Fin de la capa **gold**

## 5. Modelo

In [7]:
# ===================================================================================
# MODELO
# ===================================================================================

import numpy as np 
import pandas as pd
from rapidfuzz import fuzz
from src.constants import MATRICES_ARQUETIPO

STOPWORDS = {"la", "el", "los", "las", "un", "una", "unos", "unas", "de", "del", "y", "o", "a", "en"}

def limpiar_texto(texto: str) -> str:
    """Elimina caracteres especiales, tildes simples y stopwords."""
    if not isinstance(texto, str):
        return ""
    # Minusculas y quitar caracteres no alfanumericos
    texto = texto.lower().strip()
    palabras = re.findall(r'\b\w+\b', texto)
    # Filtrar palabras vacias
    palabras_filtradas = [p for p in palabras if p not in STOPWORDS]
    return " ".join(palabras_filtradas) if palabras_filtradas else texto

# PASO 1: FILTRO
def filtro(info_usuario: dict, umbral_similitud: float = 75, df=gold_data):
    # df = pd.read_parquet("data/gold/gold_df.parquet")
    busqueda = info_usuario['busqueda']
    restricciones = info_usuario['restricciones']

    df_filtrado = df.copy()

    titulo_query = busqueda.get("titulo_aprox")
    autor_query = busqueda.get("autor")
    
    # 1. Filtro estricto de Título
    if titulo_query:
        query_limpia = limpiar_texto(titulo_query) # ej: "quijote"
        
        def evaluar_coincidencia(titulo_libro):
            titulo_limpio = limpiar_texto(str(titulo_libro)) # ej: "don quijote de la mancha"
            
            palabras_query = set(query_limpia.split())
            palabras_titulo = set(titulo_limpio.split())
            
            # REGLA 1: Si todas las palabras clave buscadas están presentes en el título -> Pasa (Score 100)
            if palabras_query and palabras_query.issubset(palabras_titulo):
                return 100.0
            
            # REGLA 2: Si la cadena buscada está contenida como subcadena -> Pasa (Score 100)
            if query_limpia in titulo_limpio:
                return 100.0

            # REGLA 3: Si no es contención exacta, evaluar por similitud difusa (typos)
            if not palabras_query.intersection(palabras_titulo):
                score_fuzzy = fuzz.ratio(query_limpia, titulo_limpio)
                return score_fuzzy if score_fuzzy >= 75 else 0.0
            
            return fuzz.token_set_ratio(query_limpia, titulo_limpio)

        scores_titulo = df_filtrado['titulo'].fillna("").apply(evaluar_coincidencia)
        df_filtrado = df_filtrado[scores_titulo >= umbral_similitud]
        
    # 2. Filtro de Autor
    if autor_query and not df_filtrado.empty:
        scores_autor = df_filtrado['autoria'].fillna("").astype(str).apply(
            lambda x: fuzz.partial_ratio(autor_query.lower(), x.lower())
        )
        df_filtrado = df_filtrado[scores_autor >= umbral_similitud]

    # 3. Filtros duros
    for col, rest in restricciones.items():
        if col in df_filtrado.columns and rest:
            df_filtrado = df_filtrado[df_filtrado[col].between(rest[0], rest[1])]

    return df_filtrado

# PASO 2: CÁLCULO DE PESOS
CRITERIOS = [
    "indice_portabilidad",
    "indice_compacidad",
    "indice_prestancia",
    "aparato_critico",
    "prestigio_cat"
]
def calcular_vector_ahp(matriz: np.ndarray, columnas: list = CRITERIOS):
    """
    Calcula el vector de pesos normalizado y el Ratio de Consistencia (CR).
    """
    n = matriz.shape[0]
    
    # 1. Autovalores y autovectores
    autovalores, autovectores = np.linalg.eig(matriz)
    max_idx = np.argmax(np.real(autovalores))
    lambda_max = np.real(autovalores[max_idx])
    
    # 2. Vector propio principal normalizado (Pesos w_j)
    weights = np.real(autovectores[:, max_idx])
    weights = weights / np.sum(weights)
    
    # 3. Ratio de Consistencia (CR de Saaty)
    ci = (lambda_max - n) / (n - 1)
    ri_5 = 1.12  # Valor aleatorio de Saaty para n = 5
    cr = ci / ri_5
    
    dict_pesos = dict(zip(columnas, np.round(weights, 4)))
    
    return dict_pesos, cr

# PASO 3: MODELO TOPSIS
def ejecutar_topsis(df: pd.DataFrame, pesos_dict: dict) -> pd.DataFrame:
    df_res = df.copy()
    cols = list(pesos_dict.keys())
    
    # 1. Matriz de decisión
    X = df_res[cols].astype(float).values
    
    # 2. Normalización Vectorial
    normas = np.sqrt((X**2).sum(axis=0))
    normas[normas == 0.0] = 1.0
    X_norm = X / normas
    
    # 3. Ponderación con los pesos AHP
    weights = np.array([pesos_dict[c] for c in cols])
    X_weighted = X_norm * weights
    
    # 4. Solución Ideal Positiva (A+) e Ideal Negativa (A-)
    # 'precio' es costo (minimizar); los demás son beneficios (maximizar)
    ideal_pos = []
    ideal_neg = []
    
    for i, col in enumerate(cols):
        if col == "precio":
            ideal_pos.append(X_weighted[:, i].min())
            ideal_neg.append(X_weighted[:, i].max())
        else:
            ideal_pos.append(X_weighted[:, i].max())
            ideal_neg.append(X_weighted[:, i].min())
            
    ideal_pos = np.array(ideal_pos)
    ideal_neg = np.array(ideal_neg)
    
    # 5. Distancias Euclídeas
    d_pos = np.sqrt(((X_weighted - ideal_pos)**2).sum(axis=1))
    d_neg = np.sqrt(((X_weighted - ideal_neg)**2).sum(axis=1))
    
    # 6. Cercanía Relativa (Score TOPSIS de 0 a 1)
    df_res["score_topsis"] = d_neg / (d_pos + d_neg)
    
    return df_res.sort_values(by="score_topsis", ascending=False)

def aplicar_bonificaciones_usuario(df_ranking: pd.DataFrame, busqueda: dict) -> pd.DataFrame:
    if df_ranking.empty:
        return df_ranking

    df_res = df_ranking.copy()
    bonus = np.ones(len(df_res))

    # Acceso directo a claves del diccionario
    ed_pref = busqueda["ed_preferida"] if "ed_preferida" in busqueda else busqueda.get("editorial", "")
    enc_pref = busqueda["enc_preferida"] if "enc_preferida" in busqueda else busqueda.get("encuadernacion", "")

    # Aplicar Bonus de Editorial (+15%)
    if ed_pref:
        term_ed = str(ed_pref).lower().strip()
        mask_ed = df_res['editorial'].fillna("").astype(str).str.lower().str.contains(term_ed, regex=False)
        bonus += np.where(mask_ed, 0.15, 0.0)

    # Aplicar Bonus de Encuadernación (+10%)
    if enc_pref:
        term_enc = str(enc_pref).lower().strip()
        mask_enc = df_res['encuadernacion'].fillna("").astype(str).str.lower().str.contains(term_enc, regex=False)
        bonus += np.where(mask_enc, 0.10, 0.0)

    # Calcular Score Final
    df_res["score_topsis_base"] = df_res["score_topsis"]
    df_res["score_topsis"] = (df_res["score_topsis"] * bonus).clip(upper=1.0)
    
    return df_res.sort_values(by="score_topsis", ascending=False)



def recomendar_ediciones(
    info_usuario: dict, 
    matriz_ahp: np.ndarray = None,
) -> tuple[pd.DataFrame, dict, float]:

    arquetipo = info_usuario['perfil']['arquetipo']
    
    # 1. Filtro estricto por texto / restricciones
    df_filtrado = filtro(info_usuario)
    if df_filtrado.empty:
        return pd.DataFrame(), {}, 0.0

    # 2. Selección de matriz y cálculo AHP
    if matriz_ahp is not None:
        matriz = matriz_ahp
    elif arquetipo and arquetipo in MATRICES_ARQUETIPO:
        matriz = MATRICES_ARQUETIPO[arquetipo]
    else:
        matriz = MATRICES_ARQUETIPO["lectura_general"]

    pesos_dict, cr = calcular_vector_ahp(matriz, CRITERIOS)

    # 3. Ranking base con TOPSIS (evalúa solo métricas físicas/calidad)
    df_ranking = ejecutar_topsis(df_filtrado, pesos_dict)

    # 4. Aplicar multiplicador de afinidad (Preferencias del usuario)
    perfil = info_usuario["perfil"]
    df_ranking_final = aplicar_bonificaciones_usuario(df_ranking, perfil['flags_adicionales'])

    return df_ranking_final, pesos_dict, cr

In [ ]:
# Ejemplo de entrada desde la app
info_busqueda = {
    "busqueda": {
        "titulo_aprox": None,
        "autor": "Elvira Sastre",
        "categorias": ['Literatura'],
        "subcategorias": []
    },
    "restricciones": {
        "precio": [0, 50]  # Rango de precio
    },
    'perfil': {
        "arquetipo": "lectura_general", # estudio_investigacion, lectura_general, coleccion_regalo, escolar_juvenil
        "flags_adicionales": {
        "es_para_regalo": False,
        "prefiere_ilustrado": False,
        "ed_preferida": None,
        "col_preferida": None,
        "enc_preferida": None
        }
    }
}

# Ejecución usando un arquetipo (ej: "estudiante", "coleccionista")
df_res, pesos, _ = recomendar_ediciones(info_busqueda)

# # Visualizar el top 3 resultante
# if not df_res.empty:
#     print(df_res[["titulo", "editorial", "precio", "score_topsis"]].head(3))

df_res



,ean,titulo,editorial,coleccion,autoria,traductor_y_editor,otros_contribuidores,aparato_critico,tipo_aparato_critico,categorias,subcategorias,idioma_original,idioma_de_publicacion,fecha_publicacion,n_paginas,precio,alto_mm,ancho_mm,grueso,peso,encuadernacion,es_escolar,es_ibd,url,img,prestigio_antropologia,prestigio_arqueologia,prestigio_bellas_artes,prestigio_biblioteconomia,prestigio_comunicacion,prestigio_derecho,prestigio_economia,prestigio_educacion,prestigio_filosofia,prestigio_general,prestigio_geografia,prestigio_historia,prestigio_literatura,prestigio_politica,prestigio_psicologia,prestigio_sociologia,indice_portabilidad,indice_compacidad,indice_prestancia,score_colaboradores,score_critico,prestigio_cat,score_topsis,score_topsis_base
19226,9788432249273.0,Las Vulnerabilidades,Booket,Novela,[Elvira Sastre],[Sin Traducción],[],False,NaN,[Literatura],[Ficción Moderna Y Contemporánea: General Y Li...,Castellano,Castellano,2026-01-14,352.0,10.95,190.0,125.0,14.08,231.00,Rústica,False,False,https://www.todostuslibros.com/libros/las-vuln...,https://static.cegal.es/imagenes/marcadas/9788...,0.00,0.0000,0.000000,0.0,0.000000,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.707373,0.001053,0.306455,0.0,0.0,0.000000,1.000000,1.000000
78999,9788432234958.0,Días Sin Ti,Seix Barral,Biblioteca breve,[Elvira Sastre],[Sin Traducción],[],False,NaN,[Literatura],[Ficción Moderna Y Contemporánea: General Y Li...,Castellano,Castellano,2019-03-05,264.0,18.00,230.0,133.0,17.00,350.00,Rústica,False,False,https://www.todostuslibros.com/libros/dias-sin...,https://static.cegal.es/imagenes/marcadas/9788...,0.00,0.0000,0.000000,0.0,0.000000,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.621922,0.000508,0.344278,0.0,0.0,0.000000,0.699353,0.699353
78714,9788432242878.0,Las Vulnerabilidades,Seix Barral,Biblioteca breve,[Elvira Sastre],[Sin Traducción],[],False,NaN,[Literatura],[Ficción Moderna Y Contemporánea: General Y Li...,Castellano,Castellano,2024-02-14,352.0,20.90,230.0,133.0,24.00,461.00,Rústica,False,False,https://www.todostuslibros.com/libros/las-vuln...,https://static.cegal.es/imagenes/marcadas/9788...,0.00,0.0000,0.000000,0.0,0.000000,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.601681,0.000479,0.362561,0.0,0.0,0.000000,0.628137,0.628137
77669,9788408239062.0,Risas Al Punto De Sal,Planeta,No Ficción,[Raquel Sastre],[Sin Traducción],[],False,NaN,[Literatura],"[Familia Y Relaciones: Consejos Y Problemas, D...",Castellano,Castellano,2021-03-17,224.0,17.90,230.0,150.0,13.00,338.00,Rústica,False,False,https://www.todostuslibros.com/libros/risas-al...,https://static.cegal.es/imagenes/marcadas/9788...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.593484,0.000499,0.351475,0.0,0.0,0.000000,0.599299,0.599299
78821,9788432239656.0,Madrid Me Mata,Seix Barral,Los Tres Mundos,[Elvira Sastre],[Sin Traducción],[],False,NaN,[Literatura],[Ficción Moderna Y Contemporánea: General Y Li...,Castellano,Castellano,2022-02-23,304.0,19.90,220.0,145.0,25.00,602.00,Rústica,False,False,https://www.todostuslibros.com/libros/madrid-m...,https://static.cegal.es/imagenes/marcadas/9788...,0.00,0.0000,0.000000,0.0,0.000000,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.566907,0.000381,0.387724,0.0,0.0,0.000000,0.505791,0.505791
70783,9788408069584.0,La Cocina De Elvira Arús. Las Recetas Que Siem...,Planeta,Prácticos,[Elvira Arús],[Sin Traducción],[],False,NaN,[Literatura],"[Autoayuda, Desarrollo Personal Y Consejos Prá...",NaN,Castellano,2006-11-16,400.0,5.95,210.0,130.0,45.00,800.00,Rústica,False,False,https://www.todostuslibros.com/libros/la-cocin...,https://static.cegal.es/imagenes/marcadas/9788...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.552163,0.000326,0.409414,0.0,0.0,0.000000,0.453915,0.453915
43836,9788437607054.0,El Imposible Reclamo De La Eternidad,Ediciones Cátedra,Novela Cátedra,[Elvira Sagarzazu],[Sin T